# Leaf Segmentation & Plant Growth Tracking Tutorial

This notebook demonstrates a robust, step-by-step pipeline to segment leaves, isolate individual plants, and use **CIELAB Color Space** and **Principal Component Analysis (PCA)** to measure plant dimensions (length, width) and orientation in a top-down hydroponic setup.

### Pipeline Steps:
1. **Load Image & Convert Color Space**: Convert to RGB for visualization and CIELAB for color processing.
2. **Channel Separation & Visualization**: Plot L*, a*, and b* channels to understand color information.
3. **CIELAB-PCA Projection**: Project A and B color channels onto the first principal component (PC1) to isolate vegetation.
4. **Polarity Correction**: Ensure consistent positive values for green vegetation by checking correlation with the A channel.
5. **Otsu's Thresholding**: Create a binary mask from PC1.
6. **Morphological Cleaning**: Apply opening and closing operations to clean background noise and fill leaf holes.
7. **Contour Analysis & Plant Isolation**: Detect contours, filter by size, and group/sort them to identify individual plants on the grid.
8. **Shape Analysis via PCA**: For each segmented plant, run PCA on its pixel coordinates to extract length, width, and orientation angle, and visualize the principal axes.

### Step 1: Load Image & Convert Color Space

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load sample image
img_path = 'leaf-image/veg_1/IMG_7993.JPG'
img_bgr = cv2.imread(img_path)
if img_bgr is None:
    raise FileNotFoundError(f"Could not load image at {img_path}")

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2Lab)

print(f"Loaded image: {img_path}")
print(f"Image Resolution: {img_rgb.shape[1]}x{img_rgb.shape[0]}")

# Display original image
plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.title("Original RGB Image")
plt.axis('off')
plt.show()

### Step 2: CIELAB Channel Separation & Visualization

We split the image into L*, a*, and b* channels. The a* channel represents green-to-red color values, where lower (more negative) values indicate greener hues. The b* channel represents blue-to-yellow values, where higher (more positive) values indicate yellower hues.

In [ ]:
l_channel, a_channel, b_channel = cv2.split(img_lab)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(l_channel, cmap='gray')
axes[0].set_title("L* Channel (Lightness)")
axes[0].axis('off')

axes[1].imshow(a_channel, cmap='gray')
axes[1].set_title("a* Channel (Green-Red)")
axes[1].axis('off')

axes[2].imshow(b_channel, cmap='gray')
axes[2].set_title("b* Channel (Blue-Yellow)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Steps 3 & 4: CIELAB-PCA Projection & Polarity Correction

Rather than using simple, fixed color indexes, we use **Principal Component Analysis (PCA)** on the a* and b* color channels to dynamically find the axis of maximum variance. This projects the 2D color information into a 1D channel (PC1) that naturally isolates the plant pixels from the channels and background.

Because the principal direction vector is defined up to sign, we perform a **polarity correction**. We compute the correlation between the projected PC1 values and the original a* channel values. If they are positively correlated (meaning higher PC1 corresponds to redder/less green values), we multiply PC1 by -1. This guarantees that green vegetation always has the highest values, ensuring consistent thresholding.

In [ ]:
# Flatten a* and b* channels for PCA
a_flat = a_channel.astype(np.float32).reshape(-1, 1)
b_flat = b_channel.astype(np.float32).reshape(-1, 1)
color_features = np.hstack((a_flat, b_flat))

# Compute PCA using OpenCV's PCACompute2
mean, eigenvectors, eigenvalues = cv2.PCACompute2(color_features, mean=None)

# Project onto the first principal component (PC1)
projected = cv2.PCAProject(color_features, mean, eigenvectors)
pc1 = projected[:, 0].reshape(a_channel.shape)

# Polarity correction: ensure green (low values in a*) corresponds to high values in PC1
correlation = np.corrcoef(pc1.ravel(), a_channel.ravel())[0, 1]
if correlation > 0:
    pc1 = -pc1

# Scale to 0-255 range
pc1_min = pc1.min()
pc1_max = pc1.max()
pc1_scaled = np.uint8(255 * (pc1 - pc1_min) / (pc1_max - pc1_min + 1e-8))

# Visualize PC1 and its histogram
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(pc1_scaled, cmap='gray')
axes[0].set_title("Polarity-Corrected PC1 Channel")
axes[0].axis('off')

axes[1].hist(pc1_scaled.ravel(), bins=256, color='gray', range=[0, 256])
axes[1].set_title("Histogram of PC1 Values")
axes[1].set_xlabel("Pixel Intensity")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

### Steps 5 & 6: Otsu's Thresholding & Morphological Cleaning

We apply Otsu's thresholding to the scaled PC1 image to dynamically separate the foreground plants. We then clean the binary mask using morphological **opening** (removes small background noise/algae) and **closing** (fills small holes inside the leaf contours).

In [ ]:
# Otsu's Thresholding
_, binary_mask = cv2.threshold(pc1_scaled, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Morphological Cleaning
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
cleaned_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel)
cleaned_mask = cv2.morphologyEx(cleaned_mask, cv2.MORPH_CLOSE, kernel)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(binary_mask, cmap='gray')
axes[0].set_title("Otsu's Thresholded Mask")
axes[0].axis('off')

axes[1].imshow(cleaned_mask, cmap='gray')
axes[1].set_title("Morphologically Cleaned Mask")
axes[1].axis('off')

plt.tight_layout()
plt.show()

### Step 7: Contour Analysis and Plant Grid Assignment

Using the cleaned mask, we detect plant contours, filter out small noise, and group them into the 3 horizontal hydroponic channels. We sort the plants in each channel from left to right to assign persistent ID labels (11-14, 21-24, 31-34).

In [ ]:
# Find external contours
contours, _ = cv2.findContours(cleaned_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

min_area = 1000
plant_records = []

for idx, cnt in enumerate(contours):
    area = cv2.contourArea(cnt)
    if area < min_area:
        continue
    
    # Calculate centroid
    M = cv2.moments(cnt)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
    else:
        continue
        
    plant_records.append({
        'contour': cnt,
        'area': area,
        'centroid': (cx, cy)
    })

# Assign row ID based on vertical grid position (approximate channel regions)
for record in plant_records:
    cx, cy = record['centroid']
    if cy < 300:
        record['row'] = 1
    elif cy < 650:
        record['row'] = 2
    else:
        record['row'] = 3

# Sort columns (left-to-right) within each row to assign consistent plant IDs
row1 = sorted([r for r in plant_records if r['row'] == 1], key=lambda x: x['centroid'][0])
row2 = sorted([r for r in plant_records if r['row'] == 2], key=lambda x: x['centroid'][0])
row3 = sorted([r for r in plant_records if r['row'] == 3], key=lambda x: x['centroid'][0])

for col, record in enumerate(row1):
    record['col'] = col + 1
    record['id'] = 1 * 10 + (col + 1)
for col, record in enumerate(row2):
    record['col'] = col + 1
    record['id'] = 2 * 10 + (col + 1)
for col, record in enumerate(row3):
    record['col'] = col + 1
    record['id'] = 3 * 10 + (col + 1)

all_sorted_plants = row1 + row2 + row3

# Visualize the labeled plants on the image
img_labeled = img_rgb.copy()
for record in all_sorted_plants:
    cnt = record['contour']
    cx, cy = record['centroid']
    pid = record['id']
    
    # Draw contour and label
    cv2.drawContours(img_labeled, [cnt], -1, (0, 255, 0), 2)
    cv2.putText(img_labeled, f"ID: {pid}", (cx - 30, cy), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

plt.figure(figsize=(12, 10))
plt.imshow(img_labeled)
plt.title("Segmented and Identified Plants (Row/Col Grid)")
plt.axis('off')
plt.show()

print(f"Total plants detected: {len(all_sorted_plants)}")

### Step 8: Shape Analysis via PCA (Principal Component Analysis)

To track growth, we perform 2D PCA on the coordinates of all pixels belonging to each individual plant. This extracts:
1. **Length**: 2 * standard deviations along the major axis (PC1 direction).
2. **Width**: 2 * standard deviations along the minor axis (PC2 direction).
3. **Orientation Angle**: The angle of the major axis relative to the horizontal.

We draw the major and minor axis vectors (spanning $\pm 2$ standard deviations from the center) directly on the image to inspect the measurement results.

In [ ]:
def analyze_plant_shape(contour):
    # Extract coordinates of all pixels inside the contour
    mask = np.zeros(cleaned_mask.shape, dtype=np.uint8)
    cv2.drawContours(mask, [contour], -1, 255, -1)
    pts = np.column_stack(np.where(mask == 255))
    pts = pts[:, [1, 0]].astype(np.float32) # Swap to x, y format
    
    if len(pts) < 5:
        return None, None, None, None, None
    
    # Run 2D PCA on coordinates
    mean, eigenvectors, eigenvalues = cv2.PCACompute2(pts, mean=None)
    
    center = (mean[0, 0], mean[0, 1])
    sd1 = np.sqrt(max(0, eigenvalues[0, 0]))
    sd2 = np.sqrt(max(0, eigenvalues[1, 0]))
    
    length = 2.0 * sd1
    width = 2.0 * sd2
    angle = np.arctan2(eigenvectors[0, 1], eigenvectors[0, 0]) * 180.0 / np.pi
    
    return center, eigenvectors, length, width, angle

# Draw the PCA axes on the original image
img_pca = img_rgb.copy()

print("Plant ID | Centroid  | Length (px) | Width (px) | Area (px) | Orientation Angle")
print("--------------------------------------------------------------------------------")
for record in all_sorted_plants:
    cnt = record['contour']
    center, eigenvectors, length, width, angle = analyze_plant_shape(cnt)
    
    if center is None:
        continue
        
    pid = record['id']
    area = record['area']
    print(f"ID: {pid:<4} | ({int(center[0]):>3}, {int(center[1]):>3}) | {length:>11.1f} | {width:>10.1f} | {area:>9.1f} | {angle:>6.1f}°")
    
    cx, cy = int(center[0]), int(center[1])
    # Major axis endpoints (+/- 2 std dev)
    p1_major = (int(cx + eigenvectors[0, 0] * length), int(cy + eigenvectors[0, 1] * length))
    p2_major = (int(cx - eigenvectors[0, 0] * length), int(cy - eigenvectors[0, 1] * length))
    # Minor axis endpoints (+/- 2 std dev)
    p1_minor = (int(cx + eigenvectors[1, 0] * width), int(cy + eigenvectors[1, 1] * width))
    p2_minor = (int(cx - eigenvectors[1, 0] * width), int(cy - eigenvectors[1, 1] * width))
    
    # Draw center
    cv2.circle(img_pca, (cx, cy), 5, (255, 0, 0), -1)
    # Draw axes (Red = Major, Blue = Minor)
    cv2.line(img_pca, p1_major, p2_major, (255, 0, 0), 2)
    cv2.line(img_pca, p1_minor, p2_minor, (0, 0, 255), 2)
    # Label plant ID
    cv2.putText(img_pca, f"ID: {pid}", (cx - 30, cy - 20), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

plt.figure(figsize=(12, 10))
plt.imshow(img_pca)
plt.title("Hydroponic Grid Growth Analysis with PCA Axes (Red = Major, Blue = Minor)")
plt.axis('off')
plt.show()